# 01. Первый агент: state, tools и filesystem

## Что агент пока не умеет

После `00` у нас есть идея agent runtime, но ещё нет агента, который сам исследует рабочую директорию.

> Модель умеет рассуждать о коде только тогда, когда человек принёс код в контекст. Агент сам исследует окружение и решает, какие данные ему нужны.


## Какую способность добавляем

Сначала создаём минимальный Deep Agent без filesystem. Затем добавляем `FilesystemBackend` и показываем контраст до/после.

```text
Model  → решает, что делать дальше
State  → хранит сообщения и результаты действий
Tools  → позволяют воздействовать на окружение
```


## Agent loop

```text
получить state
→ вызвать модель
→ модель выбрала tool?
   → да: выполнить tool и вернуть результат в state
   → нет: завершить ответ
```

`create_deep_agent()` — это не «создай магический интеллект», а готовая сборка такого цикла.


## Workspace boundary

```text
Физический путь:
локальный каталог на компьютере

Виртуальный путь для агента:
ограниченное рабочее пространство
```

Объяснить перед кодом:

- что именно агент видит;
- откуда начинается workspace;
- может ли он писать;
- какие tools добавляются backend'ом.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

from deepagents import create_deep_agent
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

DEFAULT_MODEL = 'openrouter:tencent/hy3:free'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)

def write_text(path: str, content: str) -> Path:
    target = REPO_ROOT / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return target

def write_json(path: str, payload: dict) -> Path:
    target = REPO_ROOT / path
    target.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n')
    return target


In [ ]:
BASE_PROMPT = """\
You are OpenClaw, a messenger-first engineering agent built for a workshop.
Work from messages, inspect before acting, keep external writes gated, and never print full secrets.
"""

minimal_agent = create_deep_agent(
    model=model_name(),
    tools=[],
    system_prompt=BASE_PROMPT,
)

print(type(minimal_agent).__name__)


In [ ]:
ENTRYPOINT = "\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nfrom deepagents import create_deep_agent\nfrom dotenv import load_dotenv\n\nload_dotenv()\nDEFAULT_MODEL = \"openrouter:tencent/hy3:free\"\n\nBASE_PROMPT = \"\"\"\\\nYou are OpenClaw at stage 01. Explain your limits honestly. Before filesystem is enabled,\nyou cannot inspect the repository. With filesystem backend, cite files for repository claims.\n\"\"\"\n\nagent = create_deep_agent(\n    model=os.getenv(\"OPENCLAW_MODEL\", DEFAULT_MODEL),\n    tools=[],\n    system_prompt=BASE_PROMPT,\n)\n\n\ndef _backend():\n    root = Path(os.getenv(\"OPENCLAW_WORKSPACE\", \".\")).expanduser().resolve()\n    if os.getenv(\"OPENCLAW_ENABLE_LOCAL_SHELL\") == \"1\":\n        from deepagents.backends import LocalShellBackend\n        return LocalShellBackend(root_dir=root, virtual_mode=True, inherit_env=True, timeout=120, max_output_bytes=80_000)\n    from deepagents.backends import FilesystemBackend\n    return FilesystemBackend(root_dir=root, virtual_mode=True)\n\nagent_with_filesystem = create_deep_agent(\n    model=os.getenv(\"OPENCLAW_MODEL\", DEFAULT_MODEL),\n    tools=[],\n    system_prompt=BASE_PROMPT,\n    backend=_backend(),\n)\n"
CONFIG = {
    "dependencies": ["."],
    "graphs": {
        "openclaw_01": "./agents/openclaw_01_agent_and_filesystem.py:agent",
        "openclaw_01_fs": "./agents/openclaw_01_agent_and_filesystem.py:agent_with_filesystem",
    },
    "env": ".env",
}
print(write_text('agents/openclaw_01_agent_and_filesystem.py', ENTRYPOINT).relative_to(REPO_ROOT))
print(write_json('langgraph.openclaw_path.json', CONFIG).relative_to(REPO_ROOT))


## Проверка в LangGraph Studio

### Запрос

```text
Кто ты, какие инструменты тебе доступны и что находится в текущем репозитории?
```

### Ожидаемое поведение

1. `openclaw_01` честно говорит, что не видит содержимое репозитория.
2. `openclaw_01_fs` читает workspace и указывает файлы-основания.

### Что изменилось относительно предыдущего этапа

Появился контраст между моделью без среды и агентом с workspace.

### Текущее ограничение

Агент умеет исследовать локальную среду, но пока не взаимодействует с системами за пределами репозитория.
